# 100-d Rosenbrock: $\sum_{i=1}^{99} \left[ 10(x_{i+1} - x_i^2)^2 + (1 - x_i)^2 \right]$


In [18]:
import numpy as np

def rosenbrock_100d(x):
    """Calculate the Rosenbrock function for a 100-dimensional vector x."""
    return sum(10*(x[i+1] - x[i]**2)**2 + (1 - x[i])**2 for i in range(99))

# Initial point x0, often chosen as a starting point for optimization
x0 = np.ones(100) * -1.5  # A common choice, not the minimum but typical starting point

In [19]:
import numpy as np
from autograd import grad
from autograd import hessian
import autograd.numpy as anp
import matplotlib.pyplot as plt
import time

In [20]:
# @title Condition number = 373
#

# Compute the Hessian matrix of the Rosenbrock function
hessian_rosenbrock = hessian(rosenbrock_100d)

# Choose a point at which to evaluate the Hessian
x_point = ones_vector = np.ones(100)  # (global minimum)

# Compute the Hessian matrix at the chosen point
H = hessian_rosenbrock(x_point)

# Compute the condition number of the Hessian matrix
condition_number = np.linalg.cond(H)

# print("Hessian matrix at point", x_point, "is:")
# print(H)
print("Condition number of the Hessian matrix is:", condition_number)


Condition number of the Hessian matrix is: 373.0190670553448


In [21]:
# @title initializations

# Automatic gradient computation using Autograd
grad_rosenbrock_function = grad(rosenbrock_100d)

# Initial point for the Rosenbrock function
x0 = np.ones(100) * -1.5

num_trials = 1

# Greatest Variance

In [22]:
# @title funcs
class GradientVarianceTracker:
    def __init__(self, dim, window_size=100):
        self.dim = dim
        self.window_size = window_size
        self.grad_history = np.zeros((window_size, dim))
        self.current_index = 0
        self.full = False

    def update(self, gradient):
        self.grad_history[self.current_index] = gradient
        self.current_index = (self.current_index + 1) % self.window_size
        if self.current_index == 0:
            self.full = True

    def get_variances(self):
        if self.full:
            return np.var(self.grad_history, axis=0)
        else:
            return np.var(self.grad_history[:self.current_index], axis=0)

def generate_random_descent_vector(g, c, randtype="uniform", randnormed=True, gradnormed=True, variance_tracker=None, num_nonzero=10):
    dim = len(g)  # Dimension of the gradient vector

    # if we want to normalize the gradient
    gtemp = g
    if gradnormed:
        gtemp = g / np.linalg.norm(g)

    counter = 0
    while True:
        counter += 1

        # Generate a random vector 'u' with mostly zeros
        u = np.zeros(dim)

        if variance_tracker:
            variances = variance_tracker.get_variances()
            selected_indices = np.argsort(variances)[-num_nonzero:]
        else:
            selected_indices = np.argsort(np.abs(g))[:num_nonzero]

        if randtype == "uniform":
            u[selected_indices] = np.random.uniform(-1, 1, len(selected_indices))
        elif randtype == "normal":
            u[selected_indices] = np.random.normal(size=len(selected_indices))
        else:
            raise Exception("randtype = \'", randtype, "\' is not supported.")

        # normalize the random vector
        if randnormed:
            u /= np.linalg.norm(u)  # Normalize to make 'u' a unit vector

        # Calculate d = g ⋅ (u + 1/c * g)
        d = u + 1/c * gtemp
        check = np.dot(gtemp, d)

        # Check if the dot product is positive
        if check > 0:
            return d, counter
        else:
            return -d, counter

def random_descent(f, grad_f, x0, rand_scale=0.5, alpha=0.3, beta=0.8, precision=1e-6, max_iterations=100000, randtype="uniform", randnormed=True, gradnormed=True, window_size=100, num_nonzero=10):
    x = x0
    x_history = [x.copy()]  # Initialize history with the starting point
    f_history = [f(x)]
    start_time = time.time()
    dim = len(x0)
    variance_tracker = GradientVarianceTracker(dim, window_size)

    for iteration in range(max_iterations):
        gradient = grad_f(x)
        variance_tracker.update(gradient)
        d, _ = generate_random_descent_vector(gradient, c=rand_scale, randtype=randtype, randnormed=randnormed, gradnormed=gradnormed, variance_tracker=variance_tracker, num_nonzero=num_nonzero)

        # Backtracking line search
        t = 1.0  # Initial step size
        while f(x - t * d) > f(x) - alpha * t * np.dot(gradient, d):
            t *= beta
        x = x - t * d

        x_history.append(x.copy())  # Record the new x value
        f_history.append(f(x))
        if np.linalg.norm(gradient) < precision:
            print("Converged!!")
            break

    if np.linalg.norm(gradient) > precision:
        print("NOT Converged!!")
    time_elapsed = time.time() - start_time
    return x, f_history, np.array(x_history), iteration + 1, time_elapsed

In [23]:
# @title Rand Descent Exp (u ~ normal [-1, 1] and Normalized, g normalized)

x_opt, f_hist, x_hist, iterations, time_elapsed = random_descent(rosenbrock_100d, grad_rosenbrock_function, x0, randtype="normal", randnormed=True, gradnormed=True, rand_scale=1, window_size=10, num_nonzero=10)

print(f"Iterations: {iterations}")
print(f"Time elapsed: {time_elapsed} seconds")

Converged!!
Iterations: 5414
Time elapsed: 299.7116279602051 seconds


# Least Variance

In [24]:
# @title funcs
import numpy as np
import matplotlib.pyplot as plt
import time

class GradientVarianceTracker:
    def __init__(self, dim, window_size=100):
        self.dim = dim
        self.window_size = window_size
        self.grad_history = np.zeros((window_size, dim))
        self.current_index = 0
        self.full = False

    def update(self, gradient):
        self.grad_history[self.current_index] = gradient
        self.current_index = (self.current_index + 1) % self.window_size
        if self.current_index == 0:
            self.full = True

    def get_variances(self):
        if self.full:
            return np.var(self.grad_history, axis=0)
        else:
            return np.var(self.grad_history[:self.current_index], axis=0)

def generate_random_descent_vector(g, c, randtype="uniform", randnormed=True, gradnormed=True, variance_tracker=None, num_nonzero=10):
    dim = len(g)  # Dimension of the gradient vector

    # if we want to normalize the gradient
    gtemp = g
    if gradnormed:
        gtemp = g / np.linalg.norm(g)

    counter = 0
    while True:
        counter += 1

        # Generate a random vector 'u' with mostly zeros
        u = np.zeros(dim)

        if variance_tracker:
            variances = variance_tracker.get_variances()
            selected_indices = np.argsort(variances)[:num_nonzero]  # Select indices with the least variance
        else:
            selected_indices = np.argsort(np.abs(g))[:num_nonzero]

        if randtype == "uniform":
            u[selected_indices] = np.random.uniform(-1, 1, len(selected_indices))
        elif randtype == "normal":
            u[selected_indices] = np.random.normal(size=len(selected_indices))
        else:
            raise Exception("randtype = \'", randtype, "\' is not supported.")

        # normalize the random vector
        if randnormed:
            u /= np.linalg.norm(u)  # Normalize to make 'u' a unit vector

        # Calculate d = g ⋅ (u + 1/c * g)
        d = u + 1/c * gtemp
        check = np.dot(gtemp, d)

        # Check if the dot product is positive
        if check > 0:
            return d, counter
        else:
            return -d, counter

def random_descent(f, grad_f, x0, rand_scale=0.5, alpha=0.3, beta=0.8, precision=1e-6, max_iterations=100000, randtype="uniform", randnormed=True, gradnormed=True, window_size=100, num_nonzero=10):
    x = x0
    x_history = [x.copy()]  # Initialize history with the starting point
    f_history = [f(x)]
    start_time = time.time()
    dim = len(x0)
    variance_tracker = GradientVarianceTracker(dim, window_size)

    for iteration in range(max_iterations):
        gradient = grad_f(x)
        variance_tracker.update(gradient)
        d, _ = generate_random_descent_vector(gradient, c=rand_scale, randtype=randtype, randnormed=randnormed, gradnormed=gradnormed, variance_tracker=variance_tracker, num_nonzero=num_nonzero)

        # Backtracking line search
        t = 1.0  # Initial step size
        while f(x - t * d) > f(x) - alpha * t * np.dot(gradient, d):
            t *= beta
        x = x - t * d

        x_history.append(x.copy())  # Record the new x value
        f_history.append(f(x))
        if np.linalg.norm(gradient) < precision:
            print("Converged!!")
            break

    if np.linalg.norm(gradient) > precision:
        print("NOT Converged!!")
    time_elapsed = time.time() - start_time
    return x, f_history, np.array(x_history), iteration + 1, time_elapsed



In [25]:
x_opt, f_hist, x_hist, iterations, time_elapsed = random_descent(rosenbrock_100d, grad_rosenbrock_function, x0, randtype="normal", randnormed=True, gradnormed=True, rand_scale=1, window_size=10, num_nonzero=10)

print(f"Iterations: {iterations}")
print(f"Time elapsed: {time_elapsed} seconds")

Converged!!
Iterations: 5020
Time elapsed: 280.3643400669098 seconds
